In [ ]:
import os, torch

root_path = os.path.dirname(os.path.dirname(os.getcwd()))
os.chdir(root_path)
from modules.utils.general import get_device

### Fetch Data

In [ ]:
from modules.data_pipeline.retrieval import BaseDatasetRetriever
retriever = BaseDatasetRetriever(dataset_name="svo-probes", data_root=root_path+'/data')

### Convert to Diagrams

In [ ]:
from modules.symbolic import text_processor
functor = text_processor.TextProcessor('/Users/tls/Desktop/Work/COMP0267/assignment_5/COMP0267_CW/bobcat')
functor.text2diagram(path=root_path+'/data/svo-probes/processed', dataset=retriever.data, text_labels=retriever.text_labels)

### Compile to Quantum Circuits

In [ ]:
from modules.compilation.quantum.ansatz import IQPAnsatz
from modules.utils.general import load_pkl, store_pkl
from modules.utils.general import get_device

obmap={'n': 2, 's': 2, 'p': 2, 'out': 9}
layers = 2
ansatz = IQPAnsatz(obmap=obmap, layers=layers)
compile_id = type(ansatz).__name__ + '_' + str(obmap['n']) + '_' + str(obmap['s']) + '_' + str(obmap['p']) + '_' + str(obmap['out']) + '_' + str(layers)

In [4]:
path = os.path.join(root_path, f'data/svo-probes/processed/compiled/{compile_id}')
if not os.path.exists(path): os.mkdir(path)

splits = ['train', 'val', 'test', 'swap']
for split in splits:
    df = load_pkl(f'data/svo-probes/processed/{split}.pkl')
    compiled_df = ansatz.compile_dataset(df)
    store_pkl(compiled_df, os.path.join(path, f'{split}.pkl'))
    print(f"Compiled {split} dataset saved to {os.path.join(path, f'{split}.pkl')}")

100%|██████████| 5267/5267 [00:01<00:00, 2680.68it/s]


Compiled train dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/IQPAnsatz_2_2_2_9_2/train.pkl


100%|██████████| 1829/1829 [00:00<00:00, 2493.07it/s]


Compiled val dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/IQPAnsatz_2_2_2_9_2/val.pkl


100%|██████████| 1833/1833 [00:00<00:00, 2555.51it/s]


Compiled test dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/IQPAnsatz_2_2_2_9_2/test.pkl


100%|██████████| 95/95 [00:00<00:00, 4168.85it/s]


Compiled swap dataset saved to /Users/tls/Desktop/Work/multi_modal/qclip/data/svo-probes/processed/compiled/IQPAnsatz_2_2_2_9_2/swap.pkl


In [ ]:
from modules.data_pipeline.embeddings import ImgStream, embed_images

generator = ImgStream(f'data/svo-probes/raw/images.zip', file_type='zip')
embed_images(generator, f'data/svo-probes/processed/image_embeddings.pt', device=get_device())

### Load Towers

In [10]:
train_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/train.pkl')
val_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/val.pkl')
test_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/test.pkl')
swap_df = load_pkl(f'data/svo-probes/processed/compiled/{compile_id}/swap.pkl')

In [21]:
train_df.columns

Index(['sentence', 'corrected_sentence', 'pos_triplet', 'neg_triplet',
       'pos_url', 'neg_url', 'pos_image_id', 'neg_image_id', 'subj_neg',
       'verb_neg', 'obj_neg', 'subj', 'verb', 'obj',
       'corrected_sentence_diagram', 'corrected_sentence_einsum',
       'corrected_sentence_symbols'],
      dtype='object')

In [ ]:
from modules.models.text.einsum_quantum import QCModel
from modules.models.vision.quantum_map import QuantumFeatureMap, FrozenCLIP

BATCH_SIZE = 256
device = 'cpu' # get_device()

image_model = QuantumFeatureMap(k=9, layers=3, batch_size=BATCH_SIZE)
image_model.compile_fmap()
img_stream = torch.load(f'data/svo-probes/processed/image_embeddings.pt')
image_model.fit_image_pca(torch.stack(list(img_stream.values())).to(device))
# image_model = FrozenCLIP()

text_model = QCModel(out_q=9)
txt_stream = train_df['corrected_sentence_symbols'].tolist() + val_df['corrected_sentence_symbols'].tolist() + test_df['corrected_sentence_symbols'].tolist() + swap_df['corrected_sentence_symbols'].tolist()
text_model.from_symbols(txt_stream)
text_model.to(device)

100%|██████████| 9024/9024 [00:00<00:00, 14925.72it/s]


QCModel()

In [ ]:
from modules.data_pipeline.datasets import SVODataset, svo_collate_fn
from torch.utils.data import DataLoader
# , num_workers=4, pin_memory=True

train_dataset = SVODataset(train_df, 'data/svo-probes/processed/image_embeddings.pt')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, shuffle=True, num_workers=4, pin_memory=True)

val_dataset = SVODataset(val_df, 'data/svo-probes/processed/image_embeddings.pt')
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, num_workers=4, pin_memory=True)

test_dataset = SVODataset(test_df, 'data/svo-probes/processed/image_embeddings.pt')
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, num_workers=4, pin_memory=True)

swap_dataset = SVODataset(swap_df, 'data/svo-probes/processed/image_embeddings.pt')
swap_loader = DataLoader(swap_dataset, batch_size=BATCH_SIZE, collate_fn=svo_collate_fn, num_workers=4, pin_memory=True)

### Learn and Evaluate

In [ ]:
from modules.models.fusion.criteria import FS_InfoNCE
from modules.models.fusion.engine import ContrastiveTrainer, MMEvaluator

LR = 1e-3
optimizer = torch.optim.Adam(list(text_model.parameters()) + list(image_model.parameters()), lr=LR)
loss_fn = FS_InfoNCE()

trainer = ContrastiveTrainer(image_model, text_model, optimizer, loss_fn, device)
evaluator = MMEvaluator(image_model, text_model, device)

In [ ]:
import mlflow, time
from modules.utils.general import set_seed
from modules.models.fusion.engine import svo_mapper
from pathlib import Path

EPOCHS = 25
SEED = int.from_bytes(os.urandom(4))
set_seed(SEED)

exp_name = 'svo-probes'

run_name = f"{int(time.time())}"
checkpoint_path = f"./checkpoints/{exp_name}/{run_name}.pt"
Path(os.path.dirname(checkpoint_path)).mkdir(parents=True, exist_ok=True)
mlf_path = os.path.join(root_path, f'mlf_dbs/{exp_name}.db')


mlflow.pytorch.autolog()
mlflow.set_tracking_uri(f"sqlite:///{mlf_path}")
mlflow.set_experiment(exp_name)

with mlflow.start_run(run_name=run_name):
    mlflow.log_params({
        "epochs": EPOCHS,
        "batch_size": BATCH_SIZE,
        "learning_rate": LR,
        "temperature_parameter": loss_fn.temperature,
        "device_target": str(device),
        "seed": SEED
        })
    
    for epoch in range(EPOCHS):
        start_time = time.time()
        loss = trainer.train_epoch(train_loader, svo_mapper)
        end_time = time.time()
        
        svo_acc = evaluator.evaluate_image_choice(val_loader)
        
        mlflow.log_metric("train_loss", loss, step=epoch)
        mlflow.log_metric("svo_acc", svo_acc, step=epoch)
        print(f"Epoch {epoch} | Loss: {loss:.4f} | SVO: {svo_acc:.4f} | Time: {end_time - start_time:.2f}s")

        torch.save({
            "image": image_model.state_dict(),
            "text": text_model.state_dict(),
            "epoch": epoch,
            "train_loss": loss,
            "val_lacc": svo_acc if val_loader else None
            }, checkpoint_path)

Epoch 0 | Loss: 5.5301 | SVO: 0.4997 | Time: 31.48s
Epoch 1 | Loss: 5.5295 | SVO: 0.5052 | Time: 30.75s
Epoch 2 | Loss: 5.5290 | SVO: 0.5052 | Time: 30.61s
Epoch 3 | Loss: 5.5285 | SVO: 0.5068 | Time: 30.78s
Epoch 4 | Loss: 5.5280 | SVO: 0.5074 | Time: 31.33s
Epoch 5 | Loss: 5.5274 | SVO: 0.5052 | Time: 30.77s
Epoch 6 | Loss: 5.5268 | SVO: 0.5079 | Time: 31.76s
Epoch 7 | Loss: 5.5260 | SVO: 0.5052 | Time: 30.80s
Epoch 8 | Loss: 5.5252 | SVO: 0.5041 | Time: 30.88s
Epoch 9 | Loss: 5.5243 | SVO: 0.5068 | Time: 30.25s
Epoch 10 | Loss: 5.5233 | SVO: 0.5063 | Time: 31.12s
Epoch 11 | Loss: 5.5221 | SVO: 0.5057 | Time: 30.49s
Epoch 12 | Loss: 5.5208 | SVO: 0.5030 | Time: 31.50s
Epoch 13 | Loss: 5.5194 | SVO: 0.5057 | Time: 30.64s
Epoch 14 | Loss: 5.5179 | SVO: 0.5046 | Time: 30.43s
Epoch 15 | Loss: 5.5163 | SVO: 0.4992 | Time: 30.95s
Epoch 16 | Loss: 5.5145 | SVO: 0.4981 | Time: 30.63s
Epoch 17 | Loss: 5.5125 | SVO: 0.4975 | Time: 32.43s
Epoch 18 | Loss: 5.5105 | SVO: 0.5030 | Time: 31.51s
Epo